# Assignment 2: Data Cleaning, Analysis, and Summarization with Pandas
Comprehensive data cleaning, aggregation, and pivot table analysis on retail sales data.

In [ ]:
import io
import numpy as np
import pandas as pd

raw_csv_data = '''Order_ID,Order_Date,Region,Product_Category,Units_Sold,Unit_Price,Total_Revenue
1001,2026-01-15, North ,Electronics,5,120.0,600.0
1002,15/01/2026,South,Furniture,2,$150.00,300.0
1003,2026-01-17,east,Office Supplies,10,15.5,155.0
1004,2026-01-18,West,Electronics,-3,120.0,-360.0
1005,2026-01-19,North,furniture,4,150.0,600.0
1006,2026-01-20,South,Office Supplies,,20.0,
1007,2026-01-21,East,Electronics,8,120.0,960.0
1007,2026-01-21,East,Electronics,8,120.0,960.0
1008,invalid_date,West,Furniture,1,450.0,450.0
1009,2026-01-23,NORTH,Office Supplies,12,12.5,150.0
1010,2026-01-24,South,Electronics,6,120.0,720.0
'''

df = pd.read_csv(io.StringIO(raw_csv_data))
print('Shape:', df.shape)
df.info()
df.head()

In [ ]:
# 6 Core Cleaning Steps & Handling Values
df = df.drop_duplicates()
df['Region'] = df['Region'].str.strip().str.title()
df['Product_Category'] = df['Product_Category'].str.title()
df['Unit_Price'] = pd.to_numeric(df['Unit_Price'].astype(str).str.replace('$', '', regex=False))
df.loc[df['Units_Sold'] < 0, 'Units_Sold'] = np.nan
df['Order_Date'] = pd.to_datetime(df['Order_Date'], format='mixed', errors='coerce')
df['Units_Sold'] = df['Units_Sold'].fillna(df['Units_Sold'].median())
df['Total_Revenue'] = df['Units_Sold'] * df['Unit_Price']
df = df.dropna(subset=['Order_Date']).reset_index(drop=True)
df

In [ ]:
# GroupBy Regional Aggregations
df.groupby('Region').agg(
    Total_Revenue=('Total_Revenue', 'sum'),
    Total_Units=('Units_Sold', 'sum'),
    Average_Order_Value=('Total_Revenue', 'mean')
).round(2)

In [ ]:
# Pivot Table: Region vs Product Category
pd.pivot_table(df, values='Total_Revenue', index='Region', columns='Product_Category', aggfunc='sum', fill_value=0.0).round(2)

### Data Quality Summary
1. The original dataset contained 11 records plagued by duplicates, mixed date structures, and negative values.
2. Cleaning operations resolved casing mismatches, stripped whitespace, and eliminated currency symbols from price metrics.
3. Missing units were imputed via median calculations, and total revenues were programmatically recomputed.
4. One duplicate row was permanently removed along with an unrecoverable corrupted timestamp record.
5. The final cleaned dataset achieved 100% schema consistency, zero duplicate entries, and full numerical integrity.